<a href="https://colab.research.google.com/github/sebassanchez5680-tech/Prueba/blob/main/Sesion12_Evaluacion_Datos_Categoricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**: SEBASTIAN MIGUEL SÁNCHEZ SALAS
- **Matrícula** 271757

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [11]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [13]:
# 1.1 — Recorre todas las columnas categóricas con .value_counts() o .unique()
# para inspeccionar sus valores.
columnas_categoricas = df.select_dtypes(include=['object']).columns

for col in columnas_categoricas: #con for le decimos que recorra cada una de las columnas del df
    print(col,df[col].unique() ) #para evaluar y buscar variantes en las categorias usamos unique el cual nos devuelve una copia de cada valor que aparece en la columna




customerID ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender ['Female' 'Male']
Partner ['Yes' 'No']
Dependents ['No' 'Yes']
PhoneService ['No' 'Yes']
MultipleLines ['No phone service' 'No' 'Yes']
InternetService ['DSL' 'Fiber optic' 'No']
OnlineSecurity ['No' 'Yes' 'No internet service']
OnlineBackup ['Yes' 'No' 'No internet service']
DeviceProtection ['No' 'Yes' 'No internet service']
TechSupport ['No' 'Yes' 'No internet service']
StreamingTV ['No' 'Yes' 'No internet service']
StreamingMovies ['No' 'Yes' 'No internet service']
Contract ['Month-to-month' 'One year' 'Two year']
PaperlessBilling ['Yes' 'No']
PaymentMethod ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
TotalCharges ['29.85' '1889.5' '108.15' ... '346.45' '306.6' '6844.5']
Churn ['No' 'Yes']


**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

Despues de revisar cada una de las columnas con las categorias unicas con unique, llegue a la conclusion de que el data frame esta correctamente normalizado en cuestion de las categorias unicas ya que ninguna columna presenta valores inconsistentes

---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [14]:
df['OnlineSecurity'].value_counts()

,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [15]:
# 2.1 — Verifica: ¿las filas con 'No internet service' en OnlineSecurity
# coinciden con las filas donde InternetService == 'No'?
# (pista: cruza ambas columnas con pd.crosstab o filtrando)
#La función pd.crosstab la usamos para crear una tabla cruzada que resume la relacion y frecuencia entre dos o mas variables categoricas.
pd.crosstab(df['InternetService'], df['OnlineSecurity']) #cruzamos ambas columnas



OnlineSecurity,No,No internet service,Yes
InternetService,,,
DSL,1241,0,1180
Fiber optic,2257,0,839
No,0,1526,0


**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_ 'No internet service' no es un valor invalido ya que la tabla cruzada nos dice que realmente tiene una relacion con OnlineSecurity ya que todos de los usuarios con InternetService como 'No' tienen 'No internet service' en OnlineSecurity. Con lo cual lo correcto seria conservarlo

---
## Actividad 3 — Alta cardinalidad (25 pts)

In [16]:
# 3.1 — Calcula .nunique() para TODAS las columnas del dataset (no solo las categóricas)
# y la razón (valores únicos / total de filas) para cada una.
cardinalidad = pd.DataFrame({
    'Valores_Unicos': df.nunique(), #contamos cuantos valores unicos tiene la columna
    'Razon': df.nunique() / len(df) #usamos esta operacion para conocer el porcentaje de cardinalidad
}).sort_values(by='Valores_Unicos', ascending=False) #lo ordenamos de acuerdo a la cardinalidad mas alta

print(cardinalidad)



                  Valores_Unicos     Razon
customerID                  7043  1.000000
TotalCharges                6531  0.927304
MonthlyCharges              1585  0.225046
tenure                        73  0.010365
PaymentMethod                  4  0.000568
StreamingMovies                3  0.000426
TechSupport                    3  0.000426
OnlineBackup                   3  0.000426
StreamingTV                    3  0.000426
DeviceProtection               3  0.000426
MultipleLines                  3  0.000426
InternetService                3  0.000426
OnlineSecurity                 3  0.000426
Contract                       3  0.000426
Partner                        2  0.000284
SeniorCitizen                  2  0.000284
gender                         2  0.000284
Dependents                     2  0.000284
PhoneService                   2  0.000284
PaperlessBilling               2  0.000284
Churn                          2  0.000284


**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_ customerID es por mucho la columna que contiene mayor cardinalidad 7,043 valores unicos, con una razon del 100% de las filas. Esto es asi, porque funciona como un identificador unico para cada uno de los clientes, entonces al ser un identificador unico no sirve como variable predictoria porque siempre son diferente entre si, no tiene sentido predecir algo que sera siempre diferentes entre si.

**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [17]:
# Aplica el agrupamiento Top 10 + Otros sobre la columna de mayor cardinalidad

categorias_top10 = df['customerID'].value_counts().head(10).index.tolist()

df['customerID_agrupado'] = df['customerID'].where(df['customerID'].isin(categorias_top10), 'Otros')

df['customerID_agrupado'].value_counts()


,count
customerID_agrupado,
Otros,7033
7590-VHVEG,1
5575-GNVDE,1
9837-FWLCH,1
1699-HPSBG,1
7203-OYKCT,1
1035-IPQPU,1
7398-LXGYX,1
2823-LKABH,1


**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_ Despues de agrupar, la columna resultante no es util, esto es asi porque con el hecho de contar con una cardinalidad del 100% entonces todos los valores son distintos, practicamente no existe categorias en la columna, pero en columnas como country se cuenta con cardinalidad mas baja con lo cual existen mas categorias que pueden formarse.   

---
## Actividad 4 — Tipos de dato (30 pts)

In [18]:
df['TotalCharges'].dtype

dtype('O')

**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_

In [19]:
# 4.2 — Corrige el tipo de TotalCharges.
# Pista: pd.to_numeric() con el parámetro errors= te puede ayudar a identificar
# o manejar los valores problemáticos que encontraste en 4.1.
print(df['TotalCharges'].sort_values(ascending=False))
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce') #hacemos a total charges  se convierta en numerico y en caso de que no pueda que lo convierta en NaN


2845     999.9
3353     999.8
3686    999.45
5598     998.1
6646    997.75
         ...  
936           
753           
1340          
4380          
6754          
Name: TotalCharges, Length: 7043, dtype: object


In [20]:
# 4.3 — Convierte a category las columnas categóricas que, según lo que calculaste
# en la Actividad 3, tengan cardinalidad baja y valores fijos.
# Verifica con .dtypes que el cambio se aplicó correctamente.

cols_baja_cardinalidad = df.select_dtypes(include=['object']).columns.drop('customerID') # Identificamos las columnas que aun son texto y tienen baja cardinalidad, excluimos a customer id

# Aplicamos la conversion a las columnas que identificamos
df[cols_baja_cardinalidad] = df[cols_baja_cardinalidad].astype('category')

df.dtypes



,0
customerID,object
gender,category
SeniorCitizen,int64
Partner,category
Dependents,category
tenure,int64
PhoneService,category
MultipleLines,category
InternetService,category
OnlineSecurity,category


---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?
2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

_Tu respuesta:_

la cardinalidad me parecio mas facil, porque es sencilla de comprender, solamente es el número de valores distintos que se encuentran en las columnas, mientras que el mas dificil los valores invalidos, no tanto por su complejidad si no por el hecho de que todos los datos pueden servir de una o otra manera, el hecho de etiquetarlo como invalido nos dice como si esos datos en especifico no sirvieran para nada.


Transferencia a modelado predictivo: le diria que considerara que "customerID" es una columna que no aportara nada al modelo, mas que solo ser un identificador, por lo cual podria quitarlo de la matriz de características inmediatamente para evitar sobreajuste. mientras que totalcharges contiene valores NaN los cuales podrian considerarse como faltantes, por lo que recomendaria que eliminara dichas filas para lograr que el modelo predictivo no falle debido a dichos valores ausentes

---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.